# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. The dataset is defined by a Croissant schema and contains multiple record sets, fields, and columns.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Display main information
print(f"{dataset.metadata.name}: {dataset.metadata.description}")


## 2. Data Overview
Review available record sets, their IDs, fields, and field `@id`s, as provided by the Croissant metadata.

In [ ]:
# List all available record sets with @id and name
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rset in record_sets:
        print(f"Record set: {rset.id}")
        print(f"  Name: {rset.name}")
        print(f"  Description: {rset.description}")
        # List the fields in each record set
        print("  Fields:")
        for f in rset.fields:
            print(f"    - {f.id} (name: {f.name}, dtype: {f.data_type})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Automatically collect all record set IDs for demonstration
record_set_ids = [rset.id for rset in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with {df.shape[0]} records and {df.shape[1]} columns.")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# For demonstration, display the fields (columns) of the first loaded non-empty record set
selected_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        selected_record_set_id = k
        break
if selected_record_set_id:
    print(f"\nColumns in {selected_record_set_id}:\n", dataframes[selected_record_set_id].columns.tolist())
    dataframes[selected_record_set_id].head()
else:
    print("No non-empty record sets found.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes for further analysis.

In [ ]:
# Identify a numeric field in the selected record set (if present)
numeric_field_id = None
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    # Attempt to automatically detect a numeric field
    for col in df.columns:
        try:
            # Test if column can be converted to float
            _ = pd.to_numeric(df[col].dropna().sample(min(5, len(df))), errors='raise')
            numeric_field_id = col
            break
        except:
            continue

if selected_record_set_id and numeric_field_id:
    # Convert to float
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].dropna().mean()  # Use mean as a threshold for demo
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Auto-detect a categorical field for grouping, prefer 'description'-like columns
    preferred_names = [c for c in df.columns if ('name' in c.lower() or 'desc' in c.lower())]
    group_field = preferred_names[0] if preferred_names else None
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA or no data loaded.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of a numeric field (if available) and its grouped means by a categorical feature.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Grouped bar plot if group_field exists
    if 'group_field' in locals() and group_field:
        group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        group_means.plot(kind='bar', figsize=(8, 4))
        plt.title(f"Mean {numeric_field_id} by {group_field} (top 10)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field data to visualize.")


## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load and explore a mass spectrometry protein dataset. We accessed record sets and fields via their `@id`s, loaded records into pandas DataFrames, performed basic EDA with field normalization, and visualized distributions and group means.

You can extend this notebook with advanced preprocessing, statistical analysis, or domain-specific visualizations as needed.